In [5]:
from pymongo import MongoClient
uri = "mongodb://localhost:27017/"
def execute(callable):
    try:
        client = MongoClient(uri)
        callable(client)

        client.close()
    except Exception as e:
        raise Exception("Unable to find the document due to the following error: ", e)

def find(cursor):
    for c in cursor:
        print(c)

def aggregate(cursor):
    for c in cursor:
        print(c)

# Example 1:

- Find products where productLine is Motorcycles and buyPrice is less than 60

- Using `find`

In [ ]:
execute(lambda client: find(client.classic.products.find({"product.Line":"Motorcycles","buyPrice":{"$lt":60}})))

- Using aggregate
    - `{ $match: { <query predicate> } }`

```python
execute(lambda client: aggregate(client.classic.products.aggregate([{"$match":{"product.Line":"Motorcycles","buyPrice":{"$lt":60}}}])))
```

In [ ]:
execute(lambda client: aggregate(client.classic.products.aggregate([{"$match":{"product.Line":"Motorcycles","buyPrice":{"$lt":60}}}])))

# Example 2:

- Find product name, buyPrice, productLine where productLine is Motorcycles and buyPrice is less than 60

- Using aggregate
    - `{ $match: { <query predicate> } }`
    - `{ $project: { <specification(s)> } }`

In [ ]:
pipelines = []
pipelines.append({"$match":{"product.Line":"Motorcycles","buyPrice":{"$lt":60}}})
pipelines.append({"$project":{"product.Name":True,"buyPrice":True,"product.Line":True, "_id": False}})
execute(lambda client: aggregate(client.classic.products.aggregate(pipelines)))

# Example 3:

- Find the 3 cheapest motorcycles under $60 and show me their name, price, and category

- Using aggregate
    - `{ $match: { <query predicate> } }`
    - `{ $project: { <specification(s)> } }`
    - `{ $sort: { <field1>: <sort order>, <field2>: <sort order> ... } }`
    - `{ $limit: <positive 64-bit integer> }`



In [ ]:
pipelines = []
pipelines.append({"$match":{"product.Line":"Motorcycles","buyPrice":{"$lt":60}}})
pipelines.append({"$project":{"product.Name":True,"buyPrice":True,"product.Line":True, "_id": False}})
pipelines.append({"$sort":{"buyPrice":1}})
pipelines.append({"$limit":3})
execute(lambda client: aggregate(client.classic.products.aggregate(pipelines)))

# Example 4:

- Find product code, name, price, category and quantity order

- Using aggregate
    - `{ $lookup:{...}}`
    - `{ $project: { <specification(s)> } }`

```javascript
{
    $lookup:{
       from: <collection to join>,
       localField: <field from the input documents>,
       foreignField: <field from the documents of the "from" collection>,
       let: { <var_1>: <expression>, …, <var_n>: <expression> },
       pipeline: [ <pipeline to run> ],
       as: <output array field>
    }
}
```



In [ ]:
pipelines = []
pipelines.append({"$lookup":{"from": "products",
  "localField": "productCode",
  "foreignField": "product.Code",
  "as": "product_order"}})
pipelines.append(
{
    "$project":{
        "productCode": 1,
  "quantityOrdered": 1,
  "product_order.product.Name": 1,
  "product_order.buyPrice": 1,
  "product_order.Line": 1,
  "_id": 0
}
}
)
execute(lambda client: aggregate(client.classic.orderdetails.aggregate(pipelines)))

- $lookup returns an array (product_order), even if there's only one match.

# Example 5:

- using `$unwind` to destruct array

- Using aggregate
    - `{ $lookup:{...}}`
    - `{ $unwind: <field path> }`
    - `{ $project: { <specification(s)> } }`

In [ ]:
pipelines = []
pipelines.append({"$lookup":{"from": "products",
  "localField": "productCode",
  "foreignField": "product.Code",
  "as": "product_order"}})
pipelines.append({"$unwind":"$product_order"})
pipelines.append(
{
    "$project":{
        "productCode": 1,
  "quantityOrdered": 1,
  "product_order.product.Name": 1,
  "product_order.buyPrice": 1,
  "product_order.Line": 1,
  "_id": 0
}
}
)
execute(lambda client: aggregate(client.classic.orderdetails.aggregate(pipelines)))


# Example 6:

- using `$addFields` - Adds new fields to documents. 

- To flatten result, remove embeded object

- Using aggregate
    - `{ $lookup:{...}}`
    - `{ $unwind: <field path> }`
    - `{ $project: { <specification(s)> } }`
    - `{ $addFields: { <newField>: <expression>, ... } }`



In [ ]:
pipelines = []
pipelines.append({"$lookup":{"from": "products",
  "localField": "productCode",
  "foreignField": "product.Code",
  "as": "product_order"}})
pipelines.append({"$unwind":"$product_order"})
pipelines.append({ "$addFields": { 
        "productName": "$product_order.product.Name",
        "buyPrice": "$product_order.buyPrice",
        "Line": "$product_order.Line"
    } })
pipelines.append(
{
    "$project":{
        "_id": 0,
        "productCode": 1,
        "quantityOrdered": 1,
        "productName": 1,
        "buyPrice": 1,
        "Line": 1
}
}
)
execute(lambda client: aggregate(client.classic.orderdetails.aggregate(pipelines)))

# Example 7:

- Find product name and code from order that status is Shipped

- Using aggregate
    - `{ $lookup:{...}}`
    - `{ $match: { <query predicate> } }`
    - `{ $unwind: <field path> }`
    - `{ $addFields: { <newField>: <expression>, ... } }`
    - `{ $project: { <specification(s)> } }`

In [ ]:
embedded_pl = [
    {
        "$match": {
            "$expr": {"$eq": ["$product.Code", "$$productCode"]}  
        }
    }
]
pipelines = []
pipelines.append({"$lookup":{"from": "products",
"let": {
            "productCode": "$productCode",
        },
    "pipeline": embedded_pl,
  "as": "product_orders"}})
embedded_pl2 = [
    {
        "$match": {
            "$expr": {"$eq": ["$orderNumber", "$$o_no"]}  
        }
    }
]
pipelines.append({"$lookup":{
    "from": "orders",
    "let":{
        "o_no": "$orderNumber"
    },
    "pipeline": embedded_pl2,
    "as": "order_details"
}})
pipelines.append({"$unwind":"$product_orders"})
pipelines.append({"$unwind":"$order_details"})
pipelines.append({"$match":{"order_details.status":{"$eq":"Shipped"}}})
pipelines.append({ "$addFields": { 
        "productName": "$product_orders.product.Name",
        "productCode": "$product_orders.product.Code",
        "orderStatus": "$order_details.status",
    } })
pipelines.append(
{
    "$project":{
        "_id": 0,
        "productCode": 1,
        "productName": 1,
        "orderStatus": 1,
}
}
)
pipelines.append({
    "$limit": 5
})
pipelines.append({
    "$sort": {"productCode":1}
})
execute(lambda client: aggregate(client.classic.orderdetails.aggregate(pipelines)))

- What happend if sort before limit?

In [ ]:
embedded_pl = [
    {
        "$match": {
            "$expr": {"$eq": ["$product.Code", "$$productCode"]}  
        }
    }
]
pipelines = []
pipelines.append({"$lookup":{"from": "products",
"let": {
            "productCode": "$productCode",
        },
    "pipeline": embedded_pl,
  "as": "product_orders"}})
embedded_pl2 = [
    {
        "$match": {
            "$expr": {"$eq": ["$orderNumber", "$$o_no"]}  
        }
    }
]
pipelines.append({"$lookup":{
    "from": "orders",
    "let":{
        "o_no": "$orderNumber"
    },
    "pipeline": embedded_pl2,
    "as": "order_details"
}})
pipelines.append({"$unwind":"$product_orders"})
pipelines.append({"$unwind":"$order_details"})
pipelines.append({"$match":{"order_details.status":{"$eq":"Shipped"}}})
pipelines.append({ "$addFields": { 
        "productName": "$product_orders.product.Name",
        "productCode": "$product_orders.product.Code",
        "orderStatus": "$order_details.status",
    } })
pipelines.append(
{
    "$project":{
        "_id": 0,
        "productCode": 1,
        "productName": 1,
        "orderStatus": 1,
}
}
)
pipelines.append({
    "$sort": {"productCode":1}
})
pipelines.append({
    "$limit": 5
})
execute(lambda client: aggregate(client.classic.orderdetails.aggregate(pipelines)))

# Example 8:

- `SELECT COUNT(*) AS count FROM orders`

- using aggregate `$group` and `$sum`

In [ ]:
pipelines = []
pipelines.append({
    "$group":{
        "_id": None, # grouping all documents into a single group
        "count":{"$sum": 1}
    }
})
execute(lambda client: aggregate(client.classic.orders.aggregate(pipelines)))

- using aggregate `$count`

In [ ]:
pipelines = []
pipelines.append({
        '$count': 'count'
    })
execute(lambda client: aggregate(client.classic.orders.aggregate(pipelines)))

# Example 9:

- `SELECT SUM(priceEach * quantityOrdered) AS total FROM orderdetails`

- using aggregate `$group` , `$sum`, `$multiply`

In [ ]:
pipelines = []
pipelines.append({
    "$group":{
        "_id": None,
        "total":{"$sum": {"$multiply":["$priceEach", "$quantityOrdered"]}}
    }
})
execute(lambda client: aggregate(client.classic.orderdetails.aggregate(pipelines)))

# Example 10:

- `select o.customerNumber, sum(d.quantityOrdered * d.priceEach) as total from orders o left join orderdetails d on o.orderNumber = d.orderNumber group by o.customerNumber having total < 50000`

- using aggregate `$group` , `$sum`, `$multiply`, `$lookup`, `$match`

In [ ]:
embedded_pl = [
    {
        "$match": {
            "$expr": {"$eq": ["$orderNumber", "$$o_no"]}  
        }
    }
]
pipelines = []
pipelines.append({
    "$lookup":{
        "from": "orders",
        "let":{"o_no":"$orderNumber"},
        "pipeline": embedded_pl,
        "as": "order_details"
    }
})
pipelines.append({"$unwind":"$order_details"})
pipelines.append({ "$addFields": { 
        "customerNumber": "$order_details.customerNumber",
    } })
pipelines.append({
    "$group":{
        "_id": "$customerNumber",
        "total":{"$sum": {"$multiply":["$priceEach", "$quantityOrdered"]}}
    }
})
pipelines.append({
    "$match":{
        "total":{"$lt":50000}
    }
})
pipelines.append({"$sort":{"_id":1}})
# pipelines.append({"$count":"total"})
execute(lambda client: aggregate(client.classic.orderdetails.aggregate(pipelines)))

In [ ]:
execute(lambda client: aggregate(client.classic.products.aggregate([
    {
        '$match': {
            'product.Line': 'Motorcycles', 
            'buyPrice': {
                '$lt': 60
            }
        }
    },
    {
        '$project': {
            'product.Name': 1, 
            'buyPrice': 1, 
            'product.Line': 1, 
            '_id': 0
        }
    }
])))

{'buyPrice': Decimal128('48.81'), 'product': {'Name': '1969 Harley Davidson Ultimate Chopper', 'Line': 'Motorcycles'}}
{'buyPrice': Decimal128('24.23'), 'product': {'Name': '1936 Harley Davidson El Knucklehead', 'Line': 'Motorcycles'}}
{'buyPrice': Decimal128('32.95'), 'product': {'Name': '1957 Vespa GS150', 'Line': 'Motorcycles'}}
{'buyPrice': Decimal128('37.32'), 'product': {'Name': '1960 BSA Gold Star DBD34', 'Line': 'Motorcycles'}}
{'buyPrice': Decimal128('47.10'), 'product': {'Name': '1982 Ducati 900 Monster', 'Line': 'Motorcycles'}}
{'buyPrice': Decimal128('24.14'), 'product': {'Name': '1982 Ducati 996 R', 'Line': 'Motorcycles'}}
{'buyPrice': Decimal128('56.13'), 'product': {'Name': '1974 Ducati 350 Mk3 Desmo', 'Line': 'Motorcycles'}}
{'buyPrice': Decimal128('34.17'), 'product': {'Name': '2002 Yamaha YZR M1', 'Line': 'Motorcycles'}}


In [8]:
execute(lambda client: aggregate(client.classic.customers.aggregate([
    {
        '$project': {
            '_id': 0, 
            'customer.name': 1, 
            'customer.number': 1, 
            'customer.salesRepEmployeeNumber': 1, 
            'customer.creditLimit': 1
        }
    }, {
        '$sort': {
            'customer.creditLimit': -1
        }
    }, {
        '$limit': 5
    }
])))

{'customer': {'number': 141, 'name': 'Euro+ Shopping Channel', 'salesRepEmployeeNumber': 1370, 'creditLimit': Decimal128('227600.00')}}
{'customer': {'number': 124, 'name': 'Mini Gifts Distributors Ltd.', 'salesRepEmployeeNumber': 1165, 'creditLimit': Decimal128('210500.00')}}
{'customer': {'number': 298, 'name': 'Vida Sport, Ltd', 'salesRepEmployeeNumber': 1702, 'creditLimit': Decimal128('141300.00')}}
{'customer': {'number': 151, 'name': 'Muscle Machine Inc', 'salesRepEmployeeNumber': 1286, 'creditLimit': Decimal128('138500.00')}}
{'customer': {'number': 187, 'name': 'AV Stores, Co.', 'salesRepEmployeeNumber': 1501, 'creditLimit': Decimal128('136800.00')}}


In [9]:
execute(lambda client: aggregate(client.classic.products.aggregate([
    {
        '$lookup': {
            'from': 'orderdetails', 
            'localField': 'product.Code', 
            'foreignField': 'productCode', 
            'as': 'product_orders'
        }
    }, {
        '$project': {
            '_id': 0, 
            'product.Code': 1, 
            'product.Name': 1, 
            'product.Line': 1, 
            'quantityInStock': 1, 
            'buyPrice': 1, 
            'product_orders': 1
        }
    }, {
        '$unwind': {
            'path': '$product_orders'
        }
    }
])))

{'quantityInStock': 7933, 'buyPrice': Decimal128('48.81'), 'product': {'Code': 'S10_1678', 'Name': '1969 Harley Davidson Ultimate Chopper', 'Line': 'Motorcycles'}, 'product_orders': {'_id': ObjectId('68d7c0b3a32948f8ec0bb202'), 'orderNumber': 10107, 'productCode': 'S10_1678', 'quantityOrdered': 30, 'priceEach': Decimal128('81.35'), 'orderLineNumber': 2}}
{'quantityInStock': 7933, 'buyPrice': Decimal128('48.81'), 'product': {'Code': 'S10_1678', 'Name': '1969 Harley Davidson Ultimate Chopper', 'Line': 'Motorcycles'}, 'product_orders': {'_id': ObjectId('68d7c0b3a32948f8ec0bb276'), 'orderNumber': 10121, 'productCode': 'S10_1678', 'quantityOrdered': 34, 'priceEach': Decimal128('86.13'), 'orderLineNumber': 5}}
{'quantityInStock': 7933, 'buyPrice': Decimal128('48.81'), 'product': {'Code': 'S10_1678', 'Name': '1969 Harley Davidson Ultimate Chopper', 'Line': 'Motorcycles'}, 'product_orders': {'_id': ObjectId('68d7c0b3a32948f8ec0bb2df'), 'orderNumber': 10134, 'productCode': 'S10_1678', 'quantity

In [ ]:
execute(lambda client: aggregate(client.classic.orders.aggregate([
    {
        '$lookup': {
            'from': 'orderdetails', 
            'localField': 'orderNumber', 
            'foreignField': 'orderNumber', 
            'as': 'order_details'
        }
    }, {
        '$unwind': {
            'path': '$order_details'
        }
    }, {
        '$group': {
            '_id': '$customerNumber', 
            'total': {
                '$sum': {
                    '$multiply': [
                        '$order_details.quantityOrdered', '$order_details.priceEach'
                    ]
                }
            }
        }
    }, {
        '$addFields': {
            'customerNumber': '$_id'
        }
    }, {
        '$project': {
            '_id': 0, 
            'customerNumber': 1, 
            'total': 1
        }
    }, {
        '$match': {
            'total': {
                '$lt': 50000
            }
        }
    }, {
        '$sort':{
            'customerNumber': 1
        }
    }
])))

Exception: ('Unable to find the document due to the following error: ', OperationFailure('Unrecognized pipeline stage name: \'$order\', full error: {\'ok\': 0.0, \'errmsg\': "Unrecognized pipeline stage name: \'$order\'", \'code\': 40324, \'codeName\': \'Location40324\'}'))